# Paper figures

Reproduces the main figures of *A deep recurrent reinforcement learning approach for enhanced MPPT
in PV systems* (Applied Soft Computing, 2024) from the PV model and the saved results in
`results/`. Run from the repository root or from `notebooks/`.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.axes_grid1.inset_locator import mark_inset, zoomed_inset_axes

from mppt_env import pv_model
from mppt_env.env import MpptEnv
from mppt_env.evaluation import STATIC_PSCS
from mppt_env.profiles import irradiance_profile, temperature_profile

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})
GRID = dict(color="grey", linewidth=0.5, linestyle="--")

## Fig. 2 & 3: single PV module characteristics

In [ ]:
v = np.linspace(0, 22, 500)
i = np.maximum(pv_model.module_current(1000, 25, v, 1), 0)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(v, i, label="Current (A)")
ax.set(xlabel="Voltage (V)", ylabel="Current (A)", ylim=(0, None))
ax2 = ax.twinx()
ax2.plot(v, i * v, color="tab:orange", label="Power (W)")
ax2.set(ylabel="Power (W)", ylim=(0, None))
ax.grid(True, **GRID)
fig.legend(loc="upper right", bbox_to_anchor=(0.85, 0.88))
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for g in (200, 400, 600, 800, 1000):
    p = np.maximum(pv_model.module_current(g, 25, v, 1), 0) * v
    axes[0].plot(v, p, label=f"{g} W/m$^2$")
for t in (10, 20, 30, 40, 50):
    p = np.maximum(pv_model.module_current(1000, t, v, 1), 0) * v
    axes[1].plot(v, p, label=f"{t} °C")
for ax, title in zip(axes, ["(a) T = 25 °C", "(b) G = 1000 W/m$^2$"]):
    ax.set(xlabel="Voltage (V)", ylabel="Power (W)", title=title, xlim=(0, 25), ylim=(0, None))
    ax.grid(True, **GRID)
    ax.legend()
plt.show()

## Fig. 7: P-V characteristics of the array under the three static PSCs

In [ ]:
v = np.arange(0, 175, 0.1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, psc in zip(axes, STATIC_PSCS):
    _, _, p = pv_model.array_output(1000, 25, v, psc)
    pmax, vmax = pv_model.gmpp(1000, 25, psc, v)
    ax.plot(v, p)
    ax.plot(vmax, pmax, "o", color="tab:red")
    ax.annotate(f"({vmax:.2f}, {pmax:.0f})", (vmax, pmax), textcoords="offset points", xytext=(-30, 8))
    ax.set(title=f"PSC {psc}", xlabel="Voltage (V)", ylabel="Power (W)", ylim=(0, pmax * 1.15))
    ax.grid(True, **GRID)
plt.show()

## Fig. 9: power and reward for PSC (6, 2, 10) at 25 °C and 1000 W/m$^2$

In [ ]:
env = MpptEnv({"temperature": 25, "irradiance": 1000, "shading": (6, 2, 10), "max_steps": 2120})
env.reset()
volts, power, reward = [], [], []
for _ in range(2120):
    _, r, _, info = env.step([0.1])
    volts.append(info["Voltage"]); power.append(info["Power"]); reward.append(r)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(volts, power, label="Power (W)")
ax.set(xlabel="Voltage (V)", ylabel="Power (W)")
ax2 = ax.twinx()
ax2.plot(volts, reward, color="tab:orange", label="Reward")
ax2.set(ylabel="Reward")
fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.88))
plt.show()

## Fig. 14: episode reward during training

In [ ]:
progress = pd.read_csv("results/training/ppo_lstm_progress.csv")
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(progress["training_iteration"], progress["episode_reward_mean"], label="Mean reward")
ax.plot(progress["training_iteration"], progress["episode_reward_max"], label="Maximum reward")
ax.set(xlabel="Iteration", ylabel="Reward", xlim=(0, 300), ylim=(0, 500))
ax.grid(True, **GRID)
ax.legend(loc="lower right")
plt.show()

## Figs. 16, 18, 19, 20: MPPT performance

`results/paper/<scenario>/<a>-<b>-<c>.csv` holds the per-step power of every algorithm as reported in
the paper. Accuracy is Eq. (9), averaged after the first 200 steps.

In [ ]:
STYLE = {
    "gmpp": dict(label="GMPP", color="#d62728"),
    "po": dict(label="P&O", color="#ff7f0e"),
    "ddpg": dict(label="DDPG", color="#1f77b4"),
    "ppo_ic": dict(label="PPO+IC", color="#9467bd"),
    "ppo_lstm": dict(label="PPO-LSTM", color="#2ca02c"),
}

def plot_scenario(scenario, title, zoom=None):
    fig, axes = plt.subplots(3, 1, figsize=(8, 15))
    for ax, psc in zip(axes, STATIC_PSCS):
        df = pd.read_csv(f"results/paper/{scenario}/{'-'.join(map(str, psc))}.csv")
        for col in df.columns[1:]:
            ax.plot(df["step"], df[col], linewidth=1.0, alpha=0.8, **STYLE[col])
        ax.set(title=f"{title}: PSC {psc}", xlabel="Time Steps", ylabel="Power (W)",
               xlim=(0, len(df)), ylim=(0, df["gmpp"].max() * 1.05))
        ax.grid(True, **GRID)
        ax.legend(loc="lower right")
        if zoom:
            (x0, x1), scale = zoom
            axins = zoomed_inset_axes(ax, scale, loc="center")
            for col in df.columns[1:]:
                axins.plot(df["step"], df[col], linewidth=1.0, alpha=0.8, color=STYLE[col]["color"])
            gm = df["gmpp"].iloc[x1]
            axins.set(xlim=(x0, x1), ylim=(gm * 0.97, gm * 1.005))
            mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5")
        acc = {STYLE[c]["label"]: round((df[c][200:] / df["gmpp"][200:]).mean() * 100, 2)
               for c in df.columns[2:]}
        print(f"{scenario} {psc} accuracy (%):", acc)
    plt.show()

plot_scenario("static", "Static", zoom=((80, 130), 8))

In [ ]:
plot_scenario("irradiance", "Change in irradiance")

In [ ]:
plot_scenario("temperature", "Change in temperature")

In [ ]:
plot_scenario("irradiance_temperature", "Changes in irradiance and temperature")

## Fig. 17: dynamic environmental profiles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(irradiance_profile()); axes[0].set(xlabel="Time Step", ylabel="Irradiance (W/m$^2$)")
axes[1].plot(temperature_profile()); axes[1].set(xlabel="Time Step", ylabel="Temperature (°C)")
for ax in axes:
    ax.grid(True, **GRID)
plt.show()